# Chicago Traffic Crash Analysis – Task 1: Data Understanding & Preparation

**Goal:** Identify high-risk crash locations and their correlation with seasonal patterns, weather, traffic signal availability, speed limits, and crash severity.

---

## 📊 Task 1.1 – Data Understanding

We will explore the dataset to assess data quality, understand variable distributions, and detect correlations between key features.


In [49]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
# Import libraries
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from scipy.stats import zscore

# Plotting settings
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


###  Load Datasets

We'll start by loading the main `crashes.csv` dataset. This is the core dataset for our analysis and contains information on all reported incidents. We are adding `people.csv` and `vehicle.csv` and aggregating and doing the feature engineering here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Load raw datasets
crashes = pd.read_csv('/content/drive/MyDrive/Traffic_Crashes_-_Crashes_20250407.csv')
people = pd.read_csv('/content/drive/MyDrive/Traffic_Crashes_-_People_20250407.csv')
vehicles = pd.read_csv('/content/drive/MyDrive/Traffic_Crashes_-_Vehicles_20250407.csv')


Mounted at /content/drive


<ipython-input-51-3376149693>:6: DtypeWarning:

Columns (19,28) have mixed types. Specify dtype option on import or set low_memory=False.

<ipython-input-51-3376149693>:7: DtypeWarning:

Columns (20,39,40,41,43,47,48,49,52,54,57,58,60,70) have mixed types. Specify dtype option on import or set low_memory=False.



###  Dataset Overview

Let’s take an initial look at the data structure: columns, data types, and completeness.

In [ ]:
# Basic info
crashes.info()

# Show first few rows
crashes.head()


###  Data Quality Check

We assess missing values, duplicated records, and summary statistics to understand the dataset's quality.


In [ ]:
# Summary stats for numerical columns
crashes.describe()

# Missing values
missing_counts = crashes.isnull().sum()
missing_percent = (missing_counts / len(crashes)) * 100
missing_df = pd.DataFrame({'Missing Count': missing_counts, 'Missing %': missing_percent})
missing_df[missing_df["Missing Count"] > 0].sort_values(by="Missing %", ascending=False)

# Duplicated records
print(f"Duplicated rows: {crashes.duplicated().sum()}")


### Data Semantics

Here is the interpretation of selected key features in the `crashes` dataset:

- **CRASH_DATE**: Timestamp of when the crash occurred.
- **POSTED_SPEED_LIMIT**: Legal speed limit at the location (in mph).
- **TRAFFIC_CONTROL_DEVICE**: Type of traffic control (signal, sign, etc.).
- **WEATHER_CONDITION**: Reported weather during the crash.
- **LIGHTING_CONDITION**: Whether it was day, night, or dusk.
- **INJURIES_TOTAL**: Total number of injuries resulting from the crash.
- **INJURIES_FATAL**: Number of fatalities.
- **DAMAGE**: Reported damage level.
- **LATITUDE / LONGITUDE**: Geographic coordinates of the crash.
- **BEAT_OF_OCCURRENCE**: Police beat (a sub-area of a district) where the crash happened.
- **CRASH_TYPE**: Nature of the crash (rear-end, turning, head-on, etc.)
- **PRIM_CONTRIBUTORY_CAUSE**: Main cause reported by the officer.

We'll now explore the distributions of key variables to understand how they behave.


### 📊 Variable Distribution Analysis

We will analyze the distributions of key numeric and categorical variables to understand crash patterns.


In [ ]:
# Numeric columns to explore
num_cols = ["POSTED_SPEED_LIMIT", "INJURIES_TOTAL", "INJURIES_FATAL"]

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(1, 3, figsize=(12, 6)) # Adjust figsize as needed

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Loop through columns and plot on subplots
for i, col in enumerate(num_cols):
    sns.histplot(data=crashes, x=col, kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
# Numeric columns to explore
num_cols = ["POSTED_SPEED_LIMIT", "INJURIES_TOTAL", "INJURIES_FATAL"]

# Create a 2x2 grid of subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 8))  # Adjust figsize as needed

# Flatten the axes array for easier iteration
axes = axes.flatten()

# Loop through columns and plot on subplots
for i, col in enumerate(num_cols):
    sns.histplot(data=crashes, x=col, kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f'Distribution of {col}')

# Remove the extra empty subplot (if any)
if len(num_cols) < 4:
    fig.delaxes(axes[-1])

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
# Convert to datetime
crashes['CRASH_DATE'] = pd.to_datetime(crashes['CRASH_DATE'])

# Extract useful features
crashes['CRASH_HOUR'] = crashes['CRASH_DATE'].dt.hour
crashes['CRASH_DAY_OF_WEEK'] = crashes['CRASH_DATE'].dt.dayofweek  # 0=Monday
crashes['CRASH_MONTH'] = crashes['CRASH_DATE'].dt.month


In [ ]:
# Extended list of numeric columns
num_cols = [
    "POSTED_SPEED_LIMIT", "NUM_UNITS", "INJURIES_TOTAL", "INJURIES_FATAL",
    "INJURIES_INCAPACITATING", "INJURIES_NON_INCAPACITATING",
    "INJURIES_REPORTED_NOT_EVIDENT", "CRASH_HOUR", "CRASH_DAY_OF_WEEK"
]

# Plot histograms with KDE
for col in num_cols:
    sns.histplot(crashes[col].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


## Missing values

In [ ]:
## Plot missing values
plt.figure(figsize=(12,6))
sns.heatmap(crashes.isnull(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Missing value in the dataset')
plt.show()

In [ ]:
cat_cols = ["TRAFFIC_CONTROL_DEVICE", "WEATHER_CONDITION", "LIGHTING_CONDITION",
            "ROADWAY_SURFACE_COND", "CRASH_TYPE", "FIRST_CRASH_TYPE", "DAMAGE"]

for col in cat_cols:
    crashes[col].value_counts().head(10).plot(kind='bar')
    plt.title(f'Top 10 Categories in {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.show()


### Correlation Matrix

This heatmap shows the pairwise correlations between numerical features.


In [ ]:
# Compute correlation matrix
num_data = crashes[["POSTED_SPEED_LIMIT", "INJURIES_TOTAL", "INJURIES_FATAL"]].copy()
corr = num_data.corr()

# Plot heatmap
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation Matrix")
plt.show()


###  Extended Correlation Matrix

We analyze pairwise relationships between injury types, speed limit, time of crash, and number of units to identify potential high-impact factors.


In [ ]:
# Select relevant numeric features for correlation
corr_cols = [
    "POSTED_SPEED_LIMIT", "NUM_UNITS", "INJURIES_TOTAL", "INJURIES_FATAL",
    "INJURIES_INCAPACITATING", "INJURIES_NON_INCAPACITATING",
    "INJURIES_REPORTED_NOT_EVIDENT", "CRASH_HOUR", "CRASH_DAY_OF_WEEK", "CRASH_MONTH"
]

# Compute correlations
corr_matrix = crashes[corr_cols].corr()

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Extended Correlation Matrix of Crash Variables')
plt.show()


In [ ]:
# Function to map month to season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

# Apply function
crashes['SEASON'] = crashes['CRASH_MONTH'].apply(get_season)


In [ ]:
day_labels = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
crash_by_day = crashes['CRASH_DAY_OF_WEEK'].value_counts().sort_index()
crash_by_day.index = day_labels

sns.barplot(x=crash_by_day.index, y=crash_by_day.values)
plt.title('Number of Crashes by Day of the Week')
plt.xlabel('Day')
plt.ylabel('Number of Crashes')
plt.show()

In [ ]:
season_counts = crashes['SEASON'].value_counts()

sns.barplot(x=season_counts.index, y=season_counts.values,
            order=['Winter', 'Spring', 'Summer', 'Fall'], palette='Set2')
plt.title('Number of Crashes by Season')
plt.xlabel('Season')
plt.ylabel('Number of Crashes')
plt.show()


In [ ]:
type_counts = crashes['CRASH_TYPE'].value_counts().head(10)

sns.barplot(y=type_counts.index, x=type_counts.values, palette='pastel')
plt.title('Top 10 Crash Types')
plt.xlabel('Number of Crashes')
plt.ylabel('Crash Type')
plt.show()


In [ ]:
#plot crash by crash type
plt.figure(figsize=(10,6))
sns.countplot(x='CRASH_TYPE', data=crashes, palette='pastel')


In [ ]:
# Monthly crashes
crashes['CRASH_YEAR_MONTH'] = crashes['CRASH_DATE'].dt.to_period('M')
monthly_trend = crashes['CRASH_YEAR_MONTH'].value_counts().sort_index()

monthly_trend.plot(kind='line', marker='o')
plt.title('Monthly Crash Trends')
plt.xlabel('Year-Month')
plt.ylabel('Number of Crashes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# Extract year and month
crashes['CRASH_YEAR'] = crashes['CRASH_DATE'].dt.year
crashes['CRASH_MONTH'] = crashes['CRASH_DATE'].dt.month

# Group and pivot to create heatmap matrix
year_month_pivot = crashes.groupby(['CRASH_YEAR', 'CRASH_MONTH']).size().unstack(fill_value=0)

# Rename columns as month names for better readability
import calendar
year_month_pivot.columns = [calendar.month_abbr[m] for m in year_month_pivot.columns]

# Plot the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(year_month_pivot, annot=True, fmt='d', cmap='YlGnBu')
plt.title('🚘 Number of Crashes per Month per Year')
plt.xlabel('Month')
plt.ylabel('Year')
plt.tight_layout()
plt.show()


In [ ]:
# 1. Crash Severity Proxy: INJURIES_TOTAL
plt.figure(figsize=(8,4))
sns.histplot(crashes['INJURIES_TOTAL'], bins=30, kde=True)
plt.title('Distribution of Crash Severity (Injuries)')
plt.xlabel('Number of Injuries')
plt.show()

In [ ]:
# 2. Speed Limit
plt.figure(figsize=(8,4))
sns.histplot(crashes['POSTED_SPEED_LIMIT'], bins=20, kde=True)
plt.title('Posted Speed Limit Distribution')
plt.xlabel('Speed Limit (mph)')
plt.show()

In [ ]:
# 3. Weather Conditions
plt.figure(figsize=(10,5))
sns.countplot(data=crashes, y='WEATHER_CONDITION', order=crashes['WEATHER_CONDITION'].value_counts().head(10).index)
plt.title('Top Weather Conditions During Crashes')
plt.xlabel('Number of Crashes')
plt.show()


In [ ]:
# 4. Lighting Conditions
plt.figure(figsize=(10,5))
sns.countplot(data=crashes, y='LIGHTING_CONDITION', order=crashes['LIGHTING_CONDITION'].value_counts().head(10).index)
plt.title('Lighting Conditions During Crashes')
plt.xlabel('Number of Crashes')
plt.show()

In [ ]:
# 5. Traffic Control Devices
plt.figure(figsize=(10,5))
sns.countplot(data=crashes, y='TRAFFIC_CONTROL_DEVICE', order=crashes['TRAFFIC_CONTROL_DEVICE'].value_counts().head(10).index)
plt.title('Traffic Control Devices Present During Crashes')
plt.xlabel('Number of Crashes')
plt.show()

In [ ]:
# Top 5 crash types by season
top_types = crashes['CRASH_TYPE'].value_counts().head(5).index
filtered = crashes[crashes['CRASH_TYPE'].isin(top_types)]

sns.countplot(data=filtered, x='SEASON', hue='CRASH_TYPE', order=['Winter', 'Spring', 'Summer', 'Fall'])
plt.title('Top Crash Types by Season')
plt.ylabel('Crash Count')
plt.xlabel('Season')
plt.legend(title='Crash Type')
plt.show()


In [ ]:
#plot top 10 streets with most crashes
street_crashes = crashes['STREET_NAME'].value_counts().head(10)
plt.figure(figsize=(15,10))
street_crashes.plot(kind='bar')

In [ ]:

# Collect all unique top 5 crash types across years
top_crash_types_set = set()

for year in crashes['CRASH_YEAR'].unique():
    year_data = crashes[crashes['CRASH_YEAR'] == year]
    top_crashes = year_data['FIRST_CRASH_TYPE'].value_counts().head(5).index
    top_crash_types_set.update(top_crashes)

top_crash_types_all_years = sorted(top_crash_types_set)

# Create a color palette
palette = sns.color_palette("tab20", n_colors=len(top_crash_types_all_years))
color_map = dict(zip(top_crash_types_all_years, palette))

# Plotting setup
num_years = len(crashes['CRASH_YEAR'].unique())
num_rows = math.ceil(num_years / 3)

fig, axes = plt.subplots(num_rows, 3, figsize=(30, 5 * num_rows))
axes = axes.flatten()  # Flatten in case there's only 1 row

year_index = 0

for year in sorted(crashes['CRASH_YEAR'].unique()):
    ax = axes[year_index]

    year_data = crashes[crashes['CRASH_YEAR'] == year]
    top_5_crashes = year_data['FIRST_CRASH_TYPE'].value_counts().head(5).index
    filtered_data = year_data[year_data['FIRST_CRASH_TYPE'].isin(top_5_crashes)]

    sns.countplot(
        x='FIRST_CRASH_TYPE',
        data=filtered_data,
        order=top_5_crashes,
        ax=ax,
        palette={k: color_map[k] for k in top_5_crashes}
    )

    ax.set_title(f'Top 5 Crash Types in {year}')
    ax.set_xlabel('')
    ax.set_ylabel('Crashes')
    ax.set_xticklabels([''] * len(top_5_crashes))  # Hide x labels
    year_index += 1

# Hide any unused subplots
for i in range(year_index, len(axes)):
    fig.delaxes(axes[i])

# Add a legend (color key)
handles = [
    plt.Line2D([0], [0], marker='s', color=color_map[crash_type], linestyle='',
               markersize=10, label=crash_type)
    for crash_type in top_crash_types_all_years
]

fig.legend(handles=handles, loc='lower center', ncol=6, title="Crash Type")

plt.tight_layout(rect=[0, 0.08, 1, 1])  # Leave space at bottom for legend
plt.show()


### Crash Locations Map

Using latitude and longitude, we can visualize the geographical distribution of crashes.


In [ ]:
# Filter out invalid coordinates
geo_df = crashes.dropna(subset=["LATITUDE", "LONGITUDE"])

# Sample to avoid overload
sample_geo = geo_df.sample(n=5000, random_state=42)

fig = px.scatter_mapbox(sample_geo,
                        lat="LATITUDE",
                        lon="LONGITUDE",
                        color="INJURIES_TOTAL",
                        # size="INJURIES_TOTAL",
                        color_continuous_scale="Reds",
                        mapbox_style="carto-positron",
                        zoom=10,
                        title="Crash Locations with Injury Severity")
fig.show()


In [ ]:
!pip install folium


In [ ]:
import folium
from folium.plugins import HeatMap

# Filter for valid coordinates (Chicago area)
crash_locations = crashes[['LATITUDE', 'LONGITUDE']].dropna()
crash_locations = crash_locations[(crash_locations['LATITUDE'] != 0) & (crash_locations['LONGITUDE'] != 0)]

# Create base map centered around Chicago
m = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles='CartoDB positron')

# Add HeatMap
heat_data = [[row['LATITUDE'], row['LONGITUDE']] for index, row in crash_locations.iterrows()]
HeatMap(heat_data, radius=7, blur=10).add_to(m)

# Display map
m


# 2. Crashes with Injuries Only

In [ ]:
m2 = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles='CartoDB positron')

injury_crashes = crashes[
    (crashes['INJURIES_TOTAL'] > 0) &
    (crashes['LATITUDE'].notna()) &
    (crashes['LONGITUDE'].notna()) &
    (crashes['LATITUDE'] != 0) &
    (crashes['LONGITUDE'] != 0)
]

HeatMap(injury_crashes[['LATITUDE', 'LONGITUDE']].values.tolist(), radius=7, blur=10).add_to(m2)
m2


# 3. Crashes During Snowy or Rainy Weather

In [ ]:
m3 = folium.Map(location=[41.8781, -87.6298], zoom_start=11, tiles='CartoDB positron')

weather_crashes = crashes[
    (crashes['WEATHER_CONDITION'].str.contains('RAIN|SNOW', case=False, na=False)) &
    (crashes['LATITUDE'].notna()) &
    (crashes['LONGITUDE'].notna()) &
    (crashes['LATITUDE'] != 0) &
    (crashes['LONGITUDE'] != 0)
]

HeatMap(weather_crashes[['LATITUDE', 'LONGITUDE']].values.tolist(), radius=7, blur=10).add_to(m3)
m3


In [ ]:
import plotly.express as px

# Filter valid locations
plotly_crashes = crashes[
    (crashes['LATITUDE'].notna()) &
    (crashes['LONGITUDE'].notna()) &
    (crashes['LATITUDE'] != 0) &
    (crashes['LONGITUDE'] != 0)
]

# Basic crash map
fig = px.scatter_mapbox(
    plotly_crashes.sample(5000),  # limit to 5k for performance
    lat='LATITUDE',
    lon='LONGITUDE',
    color='INJURIES_TOTAL',
    hover_name='CRASH_DATE',
    hover_data={'LATITUDE': False, 'LONGITUDE': False, 'INJURIES_TOTAL': True, 'WEATHER_CONDITION': True},
    zoom=10,
    height=600,
    color_continuous_scale='Blues'
)

fig.update_layout(mapbox_style='carto-positron')
fig.update_layout(title='Crashes in Chicago (Injury Severity Color-Coded)', margin={"r":0,"t":40,"l":0,"b":0})
fig.show()


# Task 2 Overview: Clustering High-Risk Zones

In [ ]:
#updated with gem
# Aggregate Per Beat
# Ensure CRASH_DATE is datetime
crashes['CRASH_DATE'] = pd.to_datetime(crashes['CRASH_DATE'])

# Extract year and month
crashes['YEAR'] = crashes['CRASH_DATE'].dt.year
crashes['MONTH'] = crashes['CRASH_DATE'].dt.month

# Re-create incident_profiles
# The original code mixes tuple and lambda function syntax within the agg function.
# This is corrected below by consistently using tuples for all aggregations:

incident_profiles = crashes.groupby(['BEAT_OF_OCCURRENCE', 'YEAR', 'MONTH']).agg(
    total_crashes=('CRASH_RECORD_ID', 'count'),
    avg_injuries=('INJURIES_TOTAL', 'mean'),
    avg_speed_limit=('POSTED_SPEED_LIMIT', 'mean'),
    pct_weather_impact=('WEATHER_CONDITION', lambda x: x.str.contains('RAIN|SNOW', case=False, na=False).mean()),
    pct_night_crashes=('LIGHTING_CONDITION', lambda x: x.str.contains('DARK', case=False, na=False).mean())
).reset_index()

# Aggregate indicators over all months/years per BEAT
beat_profiles = incident_profiles.groupby('BEAT_OF_OCCURRENCE').agg({
    'total_crashes': 'mean',
    'avg_injuries': 'mean',
    'avg_speed_limit': 'mean',
    'pct_weather_impact': 'mean',
    'pct_night_crashes': 'mean'
}).reset_index()

## Normalize Features

In [ ]:
from sklearn.preprocessing import StandardScaler

features = beat_profiles.drop(columns=['BEAT_OF_OCCURRENCE'])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)


In [ ]:
from sklearn.cluster import KMeans
sse = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_scaled)
    sse.append(km.inertia_)

# Plot the elbow
plt.figure(figsize=(8, 4))
plt.plot(range(1, 11), sse, marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('SSE (Inertia)')
plt.title('Elbow Method For Optimal k')
plt.grid(True)
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42)
beat_profiles['cluster'] = kmeans.fit_predict(X_scaled)


In [ ]:
# Reshape for plotting
# melted = beat_profiles.melt(id_vars='cluster', var_name='feature', value_name='value')
# Drop beat column before plotting
melted = beat_profiles.drop(columns='BEAT_OF_OCCURRENCE').melt(id_vars='cluster', var_name='feature', value_name='value')


# Plot
plt.figure(figsize=(12, 6))
sns.boxplot(data=melted, x='feature', y='value', hue='cluster', palette='Set2')
plt.title('Cluster Profiles Across Features')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()



In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram

linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(10, 5))
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=False)
plt.title("Hierarchical Clustering Dendrogram")
plt.xlabel("Beats")
plt.ylabel("Distance")
plt.show()


In [ ]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=1.5, min_samples=3)
labels = db.fit_predict(X_scaled)
beat_profiles['dbscan_cluster'] = labels


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=beat_profiles['cluster'], palette='Set1')
plt.title('KMeans Cluster Visualization (PCA Reduced)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.grid(True)
plt.show()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# Standardized features (assume already used for clustering)
features = ['total_crashes', 'avg_injuries', 'avg_speed_limit',
            'pct_weather_impact', 'pct_night_crashes']

# Run t-SNE
tsne = TSNE(n_components=2, perplexity=13, random_state=42)
scaled_data = beat_profiles[features] # Assign the scaled data to scaled_data
tsne_results = tsne.fit_transform(scaled_data)

# Combine with cluster labels
tsne_df = pd.DataFrame(tsne_results, columns=['TSNE1', 'TSNE2'])
# Replace 'cluster_labels' with 'beat_profiles['cluster']' to access the cluster assignments:
tsne_df['cluster'] = beat_profiles['cluster']

# Visualize
plt.figure(figsize=(8, 6))
sns.scatterplot(data=tsne_df, x='TSNE1', y='TSNE2', hue='cluster', palette='Set1', alpha=0.8)
plt.title("t-SNE Cluster Visualization")
plt.xlabel("t-SNE Component 1")
plt.ylabel("t-SNE Component 2")
plt.legend(title='Cluster')
plt.tight_layout()
plt.show()

# Task 3 – Predictive analysis

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load datasets
crashes_df = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Crashes_-_Crashes_20250407.csv',
    parse_dates=['CRASH_DATE'],
    date_format='%m/%d/%Y %I:%M:%S %p',  # Handles date and time with AM/PM
    low_memory=False
)
people = pd.read_csv('/content/drive/MyDrive/Traffic_Crashes_-_People_20250407.csv', low_memory=False)
vehicles = pd.read_csv('/content/drive/MyDrive/Traffic_Crashes_-_Vehicles_20250407.csv', low_memory=False)

# Convert DAMAGE to numerical values
def damage_to_numeric(damage_str):
    if pd.isna(damage_str):
        return np.nan
    cleaned = str(damage_str).replace('$', '').replace(',', '')
    if '-' in cleaned:
        low, high = map(float, cleaned.split(' - '))
        return (low + high) / 2
    if 'OVER' in cleaned:
        return 1_500_000  # Domain-based approximation
    return np.nan

crashes_df['DAMAGE_NUM'] = crashes_df['DAMAGE'].apply(damage_to_numeric)

# Process people and vehicle data
people_agg = people.groupby('CRASH_RECORD_ID').agg(
    num_people=('PERSON_ID', 'count'),
    avg_age=('AGE', 'mean'),
    num_injuries=('INJURY_CLASSIFICATION', lambda x: (x == 'INJURED').sum()),
    num_fatalities=('INJURY_CLASSIFICATION', lambda x: (x == 'FATAL').sum())
).reset_index()

vehicles_agg = vehicles.groupby('CRASH_RECORD_ID').agg(
    num_vehicles=('VEHICLE_ID', 'count'),
    num_towed=('TOWED_I', lambda x: (x == 'YES').sum())
).reset_index()

# Process crashes data (for DAMAGE)
crashes_agg = crashes_df.groupby('CRASH_RECORD_ID').agg(
    avg_damage_per_crash=('DAMAGE_NUM', 'mean')  # Aggregate DAMAGE_NUM
).reset_index()

# Merge all datasets
merged = crashes_df.merge(crashes_agg, on='CRASH_RECORD_ID', how='left') \
               .merge(people_agg, on='CRASH_RECORD_ID', how='left') \
               .merge(vehicles_agg, on='CRASH_RECORD_ID', how='left')

# Create temporal features
merged['YEAR_MONTH'] = merged['CRASH_DATE'].dt.to_period('M')
merged['BEAT_YEAR_MONTH'] = merged['BEAT_OF_OCCURRENCE'].astype(str) + '_' + merged['YEAR_MONTH'].astype(str)

# Create final profiles with target variable
profile_features = merged.groupby('BEAT_YEAR_MONTH').agg(
    avg_age=('avg_age', 'mean'),
    avg_speed_limit=('POSTED_SPEED_LIMIT', 'mean'),
    total_injuries=('num_injuries', 'sum'),
    severe_injuries=('num_fatalities', 'sum'),
    avg_vehicles=('num_vehicles', 'mean'),
    avg_people=('num_people', 'mean'),
    avg_damage=('avg_damage_per_crash', 'mean')  # Target variable
).reset_index()

# Handle missing values
profile_features = profile_features.fillna(profile_features.median(numeric_only=True))

# Print summary
print(profile_features.head())

# Task 3.1 – Feature importance and Data Classification

In [ ]:
import seaborn as sns

# 1) Prepare features and target (no total_injuries anywhere)
feature_cols = [
    'avg_age',
    'avg_speed_limit',
    'severe_injuries',
    'avg_vehicles',
    'avg_people'
]
X = profile_features[feature_cols]
y = profile_features['avg_damage']

# 2) Train/test split, scaling, etc. (same as before)

# 3) Random Forest feature importances
rf = RandomForestRegressor(random_state=42)
rf.fit(X, y)  # fit on full X so importances align exactly with feature_cols

importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances.sort_values().plot(
    kind='barh',
    title='Feature Importance',
    xlabel='Importance',
    ylabel='Feature'
)
plt.show()

# If you ever need the numeric values:
#print(importances.sort_values(ascending=False))

# 4) Correlation matrix without total_injuries
#    either select only the cols you care about:
corr_matrix = profile_features[feature_cols + ['avg_damage']].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()


In [ ]:
import seaborn as sns
from sklearn.impute            import SimpleImputer
from sklearn.model_selection   import train_test_split
from sklearn.tree              import DecisionTreeClassifier
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import accuracy_score, classification_report, confusion_matrix

# 0) Discretize avg_damage into 3 equal-frequency bins: 0=low,1=med,2=high
profile = profile_features.copy()
profile['damage_class'] = pd.qcut(profile['avg_damage'],
                                  q=3,
                                  labels=False)

# 1) Features & class target
feature_cols = [
    'avg_age',
    'avg_speed_limit',
    'severe_injuries',
    'avg_vehicles',
    'avg_people'
]
X = profile[feature_cols]
y = profile['damage_class']

# 2) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3) Impute & scale (for KNN)
imp = SimpleImputer(strategy='mean')
X_train_imp = imp.fit_transform(X_train)
X_test_imp  = imp.transform(X_test)

scaler = StandardScaler()
X_train_knn = scaler.fit_transform(X_train_imp)
X_test_knn  = scaler.transform(X_test_imp)

# 4) Initialize classifiers
classifiers = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN'          : KNeighborsClassifier(),
    'Random Forest': RandomForestClassifier(random_state=42)
}

# 5) Train, predict & evaluate
results = {}
for name, clf in classifiers.items():
    Xi_train, Xi_test = (X_train_knn, X_test_knn) if name=='KNN' else (X_train_imp, X_test_imp)
    clf.fit(Xi_train, y_train)
    preds = clf.predict(Xi_test)
    acc   = accuracy_score(y_test, preds)
    results[name] = {
        'accuracy': acc,
        'report': classification_report(y_test, preds, target_names=['low','med','high'])
    }
    print(f"\n=== {name} ===")
    print(f"Accuracy: {acc:.3f}")
    print(results[name]['report'])

# 6) Confusion matrix for the best model (e.g. Random Forest)
best = 'Random Forest'
clf = classifiers[best]
cm  = confusion_matrix(y_test, clf.predict(X_test_imp))
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['low','med','high'],
            yticklabels=['low','med','high'],
            cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix: {best}')
plt.tight_layout()
plt.show()

# 3.2 Model Comparison and Predictive Analysis

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# 1) Load features.csv
df = pd.read_csv(
    'features.csv',
    dtype={
        'BEAT_OF_OCCURRENCE':'Int32',
        'Year':'Int16',
        'DAMAGE':'float32',
        'avg_age':'float32',
        'injury_rate':'float32',
        'avg_vehicle_year':'float32'
    }
)

# 2) Define X & y
feature_cols = ['avg_age','injury_rate','avg_vehicle_year'] + [f'M_{i}' for i in range(1,13)]
X = df[feature_cols]
y = df['DAMAGE']

# 3) Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4) Impute missing values
imp = SimpleImputer(strategy='mean')
X_train_imp = imp.fit_transform(X_train)
X_test_imp  = imp.transform(X_test)

# 5) Scale for KNN
scaler = StandardScaler()
X_train_knn = scaler.fit_transform(X_train_imp)
X_test_knn  = scaler.transform(X_test_imp)

# 6) Initialize models
models = {
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'KNN'          : KNeighborsRegressor(),
    'Random Forest': RandomForestRegressor(random_state=42)
}

# 7) Train, evaluate (MSE + MAE)
results = {
    'Model':[],
    'Train MSE':[], 'Test MSE':[],
    'Train MAE':[], 'Test MAE':[]
}

for name, mdl in models.items():
    # choose correct input
    if name == 'KNN':
        Xi_train, Xi_test = X_train_knn, X_test_knn
    else:
        Xi_train, Xi_test = X_train_imp, X_test_imp

    # fit & predict
    mdl.fit(Xi_train, y_train)
    y_pred_train = mdl.predict(Xi_train)
    y_pred_test  = mdl.predict(Xi_test)

    # store metrics
    results['Model'].append(name)
    results['Train MSE'].append(mean_squared_error(y_train, y_pred_train))
    results['Test MSE'].append( mean_squared_error(y_test,  y_pred_test))
    results['Train MAE'].append(mean_absolute_error(y_train, y_pred_train))
    results['Test MAE'].append( mean_absolute_error(y_test,  y_pred_test))

res_df = pd.DataFrame(results)

#Print all MSE & MAE
print("\nModel performance summary:")
print(res_df.to_string(index=False))

# 8) Plot two separate bar charts
idx   = np.arange(len(res_df))
bar_w = 0.35

# --- Chart 1: MSE ---
plt.figure(figsize=(8, 5))
plt.bar(idx - bar_w/2, res_df['Train MSE'], bar_w, label='Train MSE', alpha=0.7)
plt.bar(idx + bar_w/2, res_df['Test MSE'],  bar_w, label='Test MSE',  alpha=0.7)
plt.xlabel('Model')
plt.ylabel('Mean Squared Error')
plt.title('Model Performance: MSE')
plt.xticks(idx, res_df['Model'])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# --- Chart 2: MAE ---
plt.figure(figsize=(8, 5))
plt.bar(idx - bar_w/2, res_df['Train MAE'], bar_w, label='Train MAE', alpha=0.7)
plt.bar(idx + bar_w/2, res_df['Test MAE'],  bar_w, label='Test MAE',  alpha=0.7)
plt.xlabel('Model')
plt.ylabel('Mean Absolute Error')
plt.title('Model Performance: MAE')
plt.xticks(idx, res_df['Model'])
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor

# 1) Read & build the monthly DAMAGE series
agg = pd.read_csv(
    'aggregated.csv',
    dtype={'Year':'int','Month':'int','DAMAGE':'float32'}
)
agg['Date'] = pd.to_datetime(
    agg['Year'].astype(str).str.zfill(4) + '-' +
    agg['Month'].astype(str).str.zfill(2) + '-01'
)
ts = (
    agg
    .groupby('Date', as_index=True)['DAMAGE']
    .sum()
    .sort_index()
)

# 2) Create lag-1 through lag-12 features
df = pd.DataFrame(ts)
for lag in range(1, 13):
    df[f'lag_{lag}'] = df['DAMAGE'].shift(lag)
df.dropna(inplace=True)

# 3) Split into train / test (last 12 months for test)
train = df.iloc[:-12]
test  = df.iloc[-12:]

X_train = train.drop('DAMAGE', axis=1)
y_train = train['DAMAGE']
X_test  = test.drop('DAMAGE', axis=1)
y_test  = test['DAMAGE']

# 4) Train Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# 5) Walk-forward forecast for next 12 months
#    (so each prediction uses previously‐forecast lags)
history = ts.copy()
predictions = []
for _ in range(12):
    last_date = history.index[-1]
    # build feature vector from last 12 months
    last_vals = history.shift(1).iloc[-12:].values[::-1]
    feat = last_vals.reshape(1, -1)
    yhat = rf.predict(feat)[0]
    next_date = last_date + pd.DateOffset(months=1)
    history.loc[next_date] = yhat
    predictions.append((next_date, yhat))

# 6) Format predictions as a DataFrame
future = pd.DataFrame(predictions, columns=['Date','Forecast']).set_index('Date')
historical = ts.to_frame(name='Actual')

# 7) Plot
plt.figure(figsize=(14,5))
plt.plot(historical.index, historical['Actual'],      label='Historical', linewidth=2)
plt.plot(future.index,     future['Forecast'], 'r--', label='RF Forecast', linewidth=2)
plt.fill_between(future.index,
                 future['Forecast'] * 0.8,
                 future['Forecast'] * 1.2,
                 alpha=0.3)
plt.title('Monthly Total DAMAGE: Historical vs RF Forecast')
plt.xlabel('Date')
plt.ylabel('Total DAMAGE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# Task 4. Time series Analysis

In [ ]:
# Function to clean DAMAGE values
def clean_damage(value):
    if isinstance(value, str):  # Check if the value is a string
        value = value.replace('$', '').replace(',', '')  # Remove dollar signs and commas
        if ' - ' in value:  # Handle ranges like '500 - 1500'
            low, high = map(float, value.split(' - '))
            return (low + high) / 2  # Return the midpoint of the range
        elif 'OVER' in value:  # Handle 'OVER 1500'
            return float(value.split()[-1])  # Extract the numeric part after 'OVER'
    return np.nan  # Return NaN for invalid or missing values

# Load crashes.csv with DAMAGE as string first
crashes = pd.read_csv(
    '/content/drive/MyDrive/Traffic_Crashes_-_Crashes_20250407.csv',
    usecols=[
        'CRASH_RECORD_ID',
        'BEAT_OF_OCCURRENCE',
        'CRASH_DATE',
        'POSTED_SPEED_LIMIT',
        'DAMAGE'
    ],
    dtype={
        'BEAT_OF_OCCURRENCE': 'str',
        'POSTED_SPEED_LIMIT': 'float32',
        'DAMAGE': 'str'  # Read DAMAGE as string initially
    },
    parse_dates=['CRASH_DATE'],
    low_memory=False
)

# Applying the cleaning function to DAMAGE
crashes['DAMAGE'] = crashes['DAMAGE'].apply(clean_damage).astype('float32')

# Drop rows with invalid DAMAGE values
crashes = crashes.dropna(subset=['DAMAGE'])

# Verify the cleaned DAMAGE column
print("Cleaned DAMAGE values. Sample data:")
print(crashes[['DAMAGE']].head())

In [ ]:
import warnings
warnings.simplefilter("ignore", UserWarning)

crashes    = '/content/drive/MyDrive/Traffic_Crashes_-_Crashes_20250407.csv'
people  = '/content/drive/MyDrive/Traffic_Crashes_-_People_20250407.csv'
vehicle = '/content/drive/MyDrive/Traffic_Crashes_-_Vehicles_20250407.csv'

#AGGREGATION FUNCTION
def aggregate_crashes(chunksize=200_000):
    parts = []
    for chunk in pd.read_csv(
        crashes,
        usecols=['CRASH_RECORD_ID','CRASH_DATE','BEAT_OF_OCCURRENCE','DAMAGE'],
        dtype={'BEAT_OF_OCCURRENCE':'Int32'},
        parse_dates=['CRASH_DATE'],
        chunksize=chunksize,
        low_memory=False
    ):
        chunk['Year']  = chunk['CRASH_DATE'].dt.year.astype('Int16')
        chunk['Month'] = chunk['CRASH_DATE'].dt.month.astype('Int8')
        chunk.drop(columns='CRASH_DATE', inplace=True)

        # normalize DAMAGE
        chunk['DAMAGE'] = (
            chunk['DAMAGE'].fillna('0').astype(str)
                 .str.replace(r'[^\d]', '', regex=True)
                 .replace('', '0')
                 .astype('float32')
        )

        parts.append(
            chunk
            .groupby(['BEAT_OF_OCCURRENCE','Year','Month'], as_index=False)
            ['DAMAGE']
            .sum()
        )

    df = pd.concat(parts, ignore_index=True)
    agg = df.groupby(['BEAT_OF_OCCURRENCE','Year','Month'], as_index=False)['DAMAGE'].sum()
    agg.to_csv('aggregated.csv', index=False)
    print(f"✔ aggregated.csv ← {len(agg)} rows")


#FEATURE ENGINEERING
def create_feature_matrix():
    # load aggregated
    df = pd.read_csv(
        'aggregated.csv',
        dtype={'BEAT_OF_OCCURRENCE':'Int32','Year':'Int16','Month':'Int8','DAMAGE':'float32'},
        low_memory=False
    )

    #avg_speed_limit from crashes
    crash_cols = pd.read_csv(crashes, nrows=0).columns
    if 'SPEED_LIMIT' in crash_cols:
        spd = pd.read_csv(
            crashes,
            usecols=['CRASH_DATE','BEAT_OF_OCCURRENCE','SPEED_LIMIT'],
            parse_dates=['CRASH_DATE'],
            dtype={'BEAT_OF_OCCURRENCE':'Int32'},
            low_memory=False
        )
        spd['Year'], spd['Month'] = (
            spd['CRASH_DATE'].dt.year.astype('Int16'),
            spd['CRASH_DATE'].dt.month.astype('Int8')
        )
        spd.drop(columns='CRASH_DATE', inplace=True)

        avg_spd = (
            spd.groupby(['BEAT_OF_OCCURRENCE','Year','Month'], as_index=False)
               ['SPEED_LIMIT']
               .mean()
               .rename(columns={'SPEED_LIMIT':'avg_speed_limit'})
        )
        df = df.merge(avg_spd, on=['BEAT_OF_OCCURRENCE','Year','Month'], how='left')
    else:
        print("ℹ no SPEED_LIMIT → skipping avg_speed_limit")

    # B) people → avg_age & injury_rate
    lookup = pd.read_csv(
        crashes,
        usecols=['CRASH_RECORD_ID','CRASH_DATE','BEAT_OF_OCCURRENCE'],
        parse_dates=['CRASH_DATE'],
        dtype={'BEAT_OF_OCCURRENCE':'Int32'},
        low_memory=False
    )
    lookup['Year'], lookup['Month'] = (
        lookup['CRASH_DATE'].dt.year.astype('Int16'),
        lookup['CRASH_DATE'].dt.month.astype('Int8')
    )
    lookup.drop(columns='CRASH_DATE', inplace=True)

    ppl = pd.read_csv(
        people,
        usecols=['CRASH_RECORD_ID','AGE','INJURY_CLASSIFICATION'],
        dtype={'AGE':'float32'},
        low_memory=False
    ).merge(lookup, on='CRASH_RECORD_ID', how='left')

    sev = {
        'NO INJURY':0,'POSSIBLE INJURY':1,
        'NON‑INCAPACITATING':2,'INCAPACITATING':3,'FATAL':4
    }
    ppl['Severity'] = ppl['INJURY_CLASSIFICATION'].map(sev).fillna(0).astype('Int8')

    agg_p = (
        ppl.groupby(['BEAT_OF_OCCURRENCE','Year','Month'], as_index=False)
           .agg(avg_age=('AGE','mean'),
                injury_rate=('Severity','mean'))
    )
    df = df.merge(agg_p, on=['BEAT_OF_OCCURRENCE','Year','Month'], how='left')

    # C) vehicles → avg_vehicle_year
    vcols = pd.read_csv(vehicle, nrows=0).columns
    if 'VEHICLE_YEAR' in vcols:
        veh = pd.read_csv(
            vehicle,
            usecols=['CRASH_RECORD_ID','VEHICLE_YEAR'],
            dtype={'VEHICLE_YEAR':'Int16'},
            low_memory=False
        ).merge(lookup, on='CRASH_RECORD_ID', how='left')

        agg_v = (
            veh.groupby(['BEAT_OF_OCCURRENCE','Year','Month'], as_index=False)
               .agg(avg_vehicle_year=('VEHICLE_YEAR','mean'))
        )
        df = df.merge(agg_v, on=['BEAT_OF_OCCURRENCE','Year','Month'], how='left')
    else:
        print("ℹ no VEHICLE_YEAR → skipping vehicle features")

    # D) one‑hot months
    df = pd.get_dummies(df, columns=['Month'], prefix='M')

    df.to_csv('features.csv', index=False)
    print(f"✔ features.csv ← shape {df.shape}")
    return df


# RUN EVERYTHING
aggregate_crashes()
features = create_feature_matrix()
print(features.head())

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# 1) Load the raw aggregates (still has Year & Month)
profile_agg = pd.read_csv(
    'aggregated.csv',
    dtype={'BEAT_OF_OCCURRENCE':'Int32','Year':'Int16','Month':'Int8','DAMAGE':'float32'}
)

# 2) Build the Date at month-start
profile_agg['Date'] = pd.to_datetime(
    profile_agg['Year'].astype(str) + '-' +
    profile_agg['Month'].astype(str).str.zfill(2) + '-01',
    format='%Y-%m-%d'
)

# 3) Sort for time-series
profile_agg = profile_agg.sort_values(['BEAT_OF_OCCURRENCE','Date']).reset_index(drop=True)

# Now you can pick a single beat or aggregate city-wide:
# e.g. city-wide series:
series = profile_agg.groupby('Date')['DAMAGE'].sum().asfreq('MS')

# or for one beat, say beat 123:
# beat123 = profile_agg[profile_agg['BEAT_OF_OCCURRENCE']==123].set_index('Date')['DAMAGE'].asfreq('MS')

# 4) Split train/test & continue with SARIMAX as before
train_data = series.iloc[:-12]
test_data  = series.iloc[-12:]


model   = SARIMAX(train_data, order=(1,1,1), seasonal_order=(1,1,1,12))
results = model.fit(disp=False)
pred    = results.get_forecast(12).predicted_mean

# 5) Plot
plt.figure(figsize=(10, 5))
plt.plot(train_data, label='Train')
plt.plot(test_data, label='Test')
plt.plot(pred, label='Forecast')
plt.title('SARIMAX Forecast of Monthly Total DAMAGE')
plt.xlabel('Date (Monthly)')
plt.ylabel('Total DAMAGE ($)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing

# 1) Load & prepare the target series
agg = pd.read_csv('aggregated.csv', dtype={'Year':'int','Month':'int','DAMAGE':'float32'})
agg['Date'] = pd.to_datetime(
    agg['Year'].astype(str).str.zfill(4) + '-' +
    agg['Month'].astype(str).str.zfill(2) + '-01'
)
series = agg.set_index('Date')['DAMAGE'].resample('MS').sum()

# 2) Load exogenous features from features.csv
feat = pd.read_csv('features.csv', dtype={
    'Year':'int','avg_age':'float32','injury_rate':'float32','avg_vehicle_year':'float32'
})
# Reconstruct Month from dummy columns
month_cols = [f'M_{i}' for i in range(1,13)]
feat['Month'] = feat[month_cols].idxmax(axis=1).str.split('_').str[1].astype(int)
feat['Date'] = pd.to_datetime(
    feat['Year'].astype(str).str.zfill(4) + '-' +
    feat['Month'].astype(str).str.zfill(2) + '-01'
)
exog = feat.set_index('Date')[['avg_age','injury_rate','avg_vehicle_year']].resample('MS').mean().ffill()

# 3) Align into a single DataFrame
data = pd.DataFrame({'y': series}).join(exog, how='left').ffill()

# 4) Train/test split (last 12 months as test)
n_test = 12
train, test = data.iloc[:-n_test], data.iloc[-n_test:]
y_train, y_test = train['y'], test['y']
exog_train, exog_test = train.drop(columns='y'), test.drop(columns='y')

# 5a) Plain ARIMA
model_arima = ARIMA(y_train, order=(1,1,1))
res_arima  = model_arima.fit()
pred_arima = res_arima.forecast(steps=n_test)

# 5b) ARIMAX (SARIMAX with exogenous regressors)
model_arimax = SARIMAX(
    y_train,
    exog=exog_train,
    order=(1,1,1),
    seasonal_order=(1,1,1,12)
)
res_arimax  = model_arimax.fit(disp=False)
start = y_train.index[-1] + pd.DateOffset(months=1)
end   = y_train.index[-1] + pd.DateOffset(months=n_test)
pred_arimax = res_arimax.predict(start=start, end=end, exog=exog_test)

# 5c) Holt-Winters Exponential Smoothing
model_es = ExponentialSmoothing(y_train, trend='add', seasonal='add', seasonal_periods=12)
res_es   = model_es.fit()
pred_es  = res_es.forecast(n_test)

# 6) Plot all forecasts vs. test
plt.figure(figsize=(12,6))
plt.plot(y_train,                    label='Train')
plt.plot(y_test,   marker='o',      label='Test')
plt.plot(pred_arima.index,  pred_arima,  label='ARIMA')
plt.plot(pred_arimax.index, pred_arimax, label='ARIMAX')
plt.plot(pred_es.index,     pred_es,     label='Holt-Winters')
plt.title('Forecast Comparison: ARIMA vs ARIMAX vs Holt-Winters')
plt.xlabel('Date')
plt.ylabel('Monthly Total DAMAGE')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()